In [0]:
import pyspark.sql.functions as F
from datetime import datetime

###### Auto Loader for auto detection of new files

In [0]:
from pyspark.sql.types import *
expected_schema = StructType
(
    [StructField("Region",StringType(), True),
     StructField("Date", DateType(), True),
     StructField("SKU", StringType(), True),
     StructField("Sales_Quantity", IntegerType(), True),
     StructField("ingestion_time", TimestampType(), True),
     StructField("_rescued_data", StringType(), True)
    ]
)

In [0]:
df = (spark.readStream.format("cloudFiles")
 .option("cloudFiles.format", "csv")
 .option("cloudFiles.schemaEvolutionMode", "rescue")
 .option("header", "true")
 .option("cloudFiles.schemaLocation", "/Volumes/sales_forecast_project/landing_data/schema_volume/sales")
 .load("/Volumes/sales_forecast_project/landing_data/landing_data_volume/Sales")
 .withColumn("ingestion_time", F.current_timestamp()) 
)

In [0]:
display(df)

In [0]:
# checkpoint_time = datetime.now().strftime("%Y%m%d_%H%M%S")
# checkpoint_path = f"/Volumes/sales_forecast_project/landing_data/checkpoint_volume/sales/sales_{checkpoint_time}"

In [0]:
# %sql
# DROP TABLE IF EXISTS sales_forecast_project.bronze.sales    

In [0]:
%sql
-- DROP TABLE IF EXISTS sales_forecast_project.bronze.sales;

In [0]:
df.writeStream.format("delta").trigger(availableNow = True).option("mergeSchema", "true").option("checkpointLocation",checkpoint_path).toTable("sales_forecast_project.bronze.sales")

In [0]:
%sql
select * from  sales_forecast_project.bronze.sales

In [0]:
%sql
SELECT ingestion_time, COUNT(SKU)
FROM sales_forecast_project.bronze.sales
GROUP BY ingestion_time
ORDER BY ingestion_time;